In [127]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import pandas as pd


from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

import lightgbm as lgb
import xgboost as xgb

In [128]:
DATA_PATH = r'../data/encoded/telco_churn_encoded.csv'
# telco_churn_all_onehot.csv
DATA_CLEANED_PATH = r'../data/cleaned/cleaned_telco_churn.csv'

df =  pd.read_csv(DATA_PATH)

In [129]:

# 1. Tách biến đặc trưng (X) và biến mục tiêu (y)
# X: Bỏ cột 'customerID' (không mang giá trị học máy) và cột mục tiêu 'Churn'
# y: Cột nhãn 'Churn' (cần chuyển đổi Yes/No thành 1/0 nếu chưa làm)

# Chuyển đổi nhãn Churn từ 'Yes'/'No' sang 1/0 (nếu dữ liệu gốc vẫn là dạng chữ)
if df['Churn'].dtype == 'object':
    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

y = df['Churn'].map({'Yes': 1, 'No': 0}).astype(int)
X = df.drop(columns=['customerID', 'Churn'])

# 2. Chia tập dữ liệu thành Train và Test (ví dụ: 80% train, 20% test)
# Thêm stratify=y để đảm bảo tỷ lệ khách hàng Churn ở tập train và test tương đương nhau
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Kích thước X_train: {X_train.shape}')
print(f'Kích thước X_test: {X_test.shape}')
print(f'Kích thước y_train: {y_train.shape}')
print(f'Kích thước y_test: {y_test.shape}')

Kích thước X_train: (5625, 57)
Kích thước X_test: (1407, 57)
Kích thước y_train: (5625,)
Kích thước y_test: (1407,)


In [130]:
# # 1. Kiểm tra tỷ lệ trước khi SMOTE trên tập train
# print('Tỷ lệ nhãn trước khi SMOTE (y_train):')
# print(y_train.value_counts())

# # 2. Khởi tạo SMOTE
# # random_state giúp kết quả đồng nhất ở các lần chạy
# smote = SMOTE(random_state=42)

# # 3. Fit và Resample chỉ trên tập train
# X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# # 4. Kiểm tra tỷ lệ sau khi SMOTE
# print('\nTỷ lệ nhãn sau khi SMOTE (y_train_resampled):')
# print(y_train_resampled.value_counts())

# print(f'\nKích thước X_train sau khi SMOTE: {X_train_resampled.shape}')

In [131]:
# from lazypredict.Supervised import LazyClassifier

# clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
# models, predictions = clf.fit(
#     X_train_resampled,
#     X_test,
#     y_train_resampled,
#     y_test,
# )
# models

In [132]:
# trọng số hình phạt dựa trên tỉ lệ cân bằng của dữ liệu target
num_neg = (y_train == 0).sum()
num_pos = (y_train == 1).sum()
scale_weight = num_neg / num_pos


# ==========================================
# 3. KHỞI TẠO CÁC MÔ HÌNH CÓ TRỌNG SỐ PHẠT
# ==========================================
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200, 
        max_depth=10, 
        class_weight='balanced',  # <--- Tương đương scale_pos_weight trong Random Forest
        random_state=42, 
        n_jobs=-1
    ),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        scale_pos_weight=scale_weight, # <--- Tăng trọng số phạt lớp 1
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        scale_pos_weight=scale_weight, # <--- Tăng trọng số phạt lớp 1
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1
    )
}


In [133]:
target_names = ['No Churn (0)', 'Churn (1)']

for name, model in models.items():
    print("=" * 60)
    print(f"BAO CAO CHI TIET: {name.upper()}")
    print("=" * 60)
    
    # Fit model
    model.fit(X_train, y_train)
    
    # Dự đoán class và xác suất
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    custom_threshold = 0.6
    y_pred_custom = (y_proba >= custom_threshold).astype(int)
    # In Classification Report chuẩn
    print(f"--- Classification Report với Threshold = {custom_threshold} ---")
    print(classification_report(y_test, y_pred_custom, target_names=target_names, digits=4))
        
    # In thêm chỉ số ROC-AUC
    auc = roc_auc_score(y_test, y_proba)
    print(f"ROC-AUC Score: {auc:.4f}\n")

BAO CAO CHI TIET: RANDOM FOREST
--- Classification Report với Threshold = 0.6 ---
              precision    recall  f1-score   support

No Churn (0)     0.8724    0.8277    0.8495      1033
   Churn (1)     0.5831    0.6658    0.6217       374

    accuracy                         0.7846      1407
   macro avg     0.7278    0.7467    0.7356      1407
weighted avg     0.7955    0.7846    0.7889      1407

ROC-AUC Score: 0.8305

BAO CAO CHI TIET: LIGHTGBM
--- Classification Report với Threshold = 0.6 ---
              precision    recall  f1-score   support

No Churn (0)     0.8716    0.8015    0.8351      1033
   Churn (1)     0.5514    0.6738    0.6065       374

    accuracy                         0.7676      1407
   macro avg     0.7115    0.7377    0.7208      1407
weighted avg     0.7865    0.7676    0.7743      1407

ROC-AUC Score: 0.8260

BAO CAO CHI TIET: XGBOOST
--- Classification Report với Threshold = 0.6 ---
              precision    recall  f1-score   support

No Churn (

# tìm siêu tham số cho mô hình

In [134]:
from sklearn.model_selection import GridSearchCV
import xgboost as xgb

# 1. Khởi tạo mô hình XGBoost cơ sở
# (Sử dụng cấu hình thiết bị tối ưu mà bạn đã dùng trước đó)
xgb_model = xgb.XGBClassifier(
    tree_method='hist',
    device='cuda',  # Thay thành 'cpu' nếu máy bạn không dùng GPU
    random_state=42,
    eval_metric='logloss',
)

# 2. Định nghĩa không gian tham số cần quét (Grid Search Space)
param_grid = {
    'n_estimators': [100, 300],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5, 7],
    'min_child_weight': [1, 3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'gamma': [0, 0.1],
}

# 3. Khởi tạo GridSearchCV
# scoring='f1' hoặc 'roc_auc' tùy thuộc vào việc bạn ưu tiên F1-score của lớp Churn hay ROC-AUC tổng thể
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=5,  # Cross-validation 5 folds
    scoring='f1',  # Tối ưu hóa theo điểm F1 (hoặc đổi thành 'roc_auc')
    n_jobs=-1,  # Sử dụng tối đa tài nguyên CPU
    verbose=1,
)

# 4. Tiến hành chạy Grid Search trên tập dữ liệu đã qua xử lý và SMOTE
print('Đang tiến hành tìm kiếm tham số tối ưu bằng GridSearchCV...')
grid_search.fit(X_train, y_train)

# 5. In ra kết quả tốt nhất
print('\n--- KẾT QUẢ TỐI ƯU ---')
print('Bộ tham số tốt nhất:', grid_search.best_params_)
print('Điểm số tốt nhất (Cross-Validation):', grid_search.best_score_)

# 6. Lưu lại mô hình tốt nhất để đem đi dự đoán/đánh giá


Đang tiến hành tìm kiếm tham số tối ưu bằng GridSearchCV...
Fitting 5 folds for each of 192 candidates, totalling 960 fits


/home/loc-nguyen/PROJECT/AI/OCR/venv/lib/python3.14/site-packages/xgboost/core.py:569: UserWarning: [03:18:25] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
/home/loc-nguyen/PROJECT/AI/OCR/venv/lib/python3.14/site-packages/xgboost/core.py:569: UserWarning: [03:18:26] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matche


--- KẾT QUẢ TỐI ƯU ---
Bộ tham số tốt nhất: {'colsample_bytree': 0.8, 'gamma': 0.1, 'learning_rate': 0.1, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 100, 'subsample': 0.8}
Điểm số tốt nhất (Cross-Validation): 0.5919288415131878


In [ ]:
best_xgb = grid_search.best_estimator_
display(best_xgb)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0.1,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=1, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)
